# Movie Genre Classification from Movie Posters
End-to-end walkthrough: EDA → preprocessing → MobileNetV2 transfer learning → evaluation → prediction.

This notebook calls into the reusable modules in `src/` rather than duplicating logic, so the notebook and the production scripts (`src/train.py`, `src/evaluate.py`, etc.) always stay in sync.

**Prerequisite:** run `python src/download_images.py` from the project root at least once before this notebook (it needs internet access to fetch posters from TMDB).

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))
import config
from data_preprocessing import prepare_all, set_global_seeds, load_clean_dataframe, parse_genres
set_global_seeds()

## 1. Load and inspect the raw dataset

In [ ]:
import pandas as pd
raw_df = pd.read_csv(config.CSV_PATH, lineterminator='\n')
print(raw_df.shape)
raw_df.head()

## 2. Exploratory Data Analysis
(Full plots are generated by `python src/eda.py` into `reports/`; a quick inline summary here.)

In [ ]:
from eda import run_eda
summary = run_eda()
summary

## 3. Multi-hot genre encoding and train/val/test split (70/15/15)

In [ ]:
data = prepare_all()
print('Classes:', data['classes'])
print('Train/Val/Test sizes:', len(data['train_df']), len(data['val_df']), len(data['test_df']))

## 4. Peek at an augmented training batch

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
for images, labels in data['train_ds'].take(1):
    fig, axes = plt.subplots(1, 4, figsize=(14, 4))
    for i, ax in enumerate(axes):
        # undo MobileNetV2 preprocessing (roughly) just for display
        img = (images[i].numpy() + 1.0) / 2.0
        ax.imshow(np.clip(img, 0, 1))
        genre_idx = np.where(labels[i].numpy() == 1)[0]
        ax.set_title(', '.join([data['classes'][g] for g in genre_idx]), fontsize=8)
        ax.axis('off')
    plt.tight_layout()

## 5. Build and train the model (two-stage transfer learning)
See `src/train.py` for the full, callback-driven training loop. Below mirrors it at a smaller scale for demonstration inside the notebook.

In [ ]:
from train import build_model, compile_model, get_callbacks
import config

model, base_model = build_model(num_classes=len(data['classes']))
compile_model(model, config.STAGE1_LR)
model.summary()

In [ ]:
# Stage 1: train the head only (backbone frozen)
history_1 = model.fit(
    data['train_ds'],
    validation_data=data['val_ds'],
    epochs=config.STAGE1_EPOCHS,
    callbacks=get_callbacks(config.BEST_STAGE1_PATH),
)

In [ ]:
# Stage 2: fine-tune the top of MobileNetV2
base_model.trainable = True
for layer in base_model.layers[:config.FINE_TUNE_AT_LAYER]:
    layer.trainable = False
compile_model(model, config.STAGE2_LR)

history_2 = model.fit(
    data['train_ds'],
    validation_data=data['val_ds'],
    epochs=config.STAGE2_EPOCHS,
    callbacks=get_callbacks(config.BEST_STAGE2_PATH),
)

In [ ]:
model.save(config.MODEL_PATH)
import json
with open(config.CLASSES_PATH, 'w') as f:
    json.dump(data['classes'], f, indent=2)
print('Saved model to', config.MODEL_PATH)

## 6. Plot training curves

In [ ]:
def plot_history(h1, h2, metric):
    vals = h1.history[metric] + h2.history[metric]
    val_vals = h1.history[f'val_{metric}'] + h2.history[f'val_{metric}']
    plt.plot(vals, label='train')
    plt.plot(val_vals, label='val')
    plt.axvline(len(h1.history[metric]), color='gray', linestyle='--', label='fine-tune starts')
    plt.title(metric)
    plt.legend()
    plt.show()

plot_history(history_1, history_2, 'loss')
plot_history(history_1, history_2, 'binary_accuracy')

## 7. Evaluate on the test set

In [ ]:
from evaluate import evaluate_model
report = evaluate_model()
report['overall_metrics']

## 8. Try a prediction

In [ ]:
from predict import predict_movie_genre
sample_row = data['test_df'].iloc[0]
result = predict_movie_genre(sample_row['Local_Image_Path'])
print('Actual genres:', sample_row['genre_list'])
print('Predicted:', result['predicted_genres'])